# SmartFarm ML — Stage 01: The mindset (Label Leakage demo)
**Goal:** build a model that scores a "perfect" ~1.00 on purpose, prove it's fake,
then watch it collapse when we remove the one leaked feature.
No new data — same `irrigation_readings.csv`.

## 1. Load + drop the garbage rows (from Stage 00)

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

df = pd.read_csv("irrigation_readings.csv", parse_dates=["timestamp"])

# keep only physically valid soil readings (0..100)
df = df[df["soil_moisture"].between(0, 100)].copy()
print("rows kept:", len(df))

rows kept: 650


## 2. Build a LEAKED label on purpose
This mimics the reference repo: the label is *defined by* soil_moisture,
and then we also feed soil_moisture in as a feature. The answer is hiding in the input.

In [14]:
# a comparison on a whole column returns a True/False Series (no loop needed);
# .astype(int) turns True/False into 1/0
df["leaky_label"] = (df["soil_moisture"] < 54).astype(int)
df["leaky_label"].value_counts()

leaky_label
1    330
0    320
Name: count, dtype: int64

## 3. Features (X) and label (y) — WITH soil_moisture included
Double brackets `df[[...]]` select several columns → a DataFrame (2D, the features).
Single bracket `df[...]` selects one column → a Series (1D, the label).

In [15]:
feature_cols = ["soil_moisture", "air_humidity", "temperature"]
X = df[feature_cols]      # 2D features
y = df["leaky_label"]     # 1D label

# train_test_split returns 4 things at once; Python unpacks the tuple into 4 names.
# test_size=0.3 -> 30% held out for honest testing. random_state just makes it repeatable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

## 4. Train, then score on the held-out test set

In [16]:
# .fit() trains the model in place (like calling model.train(...) in Java).
model = DecisionTreeClassifier(max_depth=3, random_state=0)
model.fit(X_train, y_train)

# .score() returns accuracy on data the model never saw during training.
acc = model.score(X_test, y_test)
print(f"accuracy WITH soil_moisture: {acc:.3f}")   # expect ~1.00 -> RED FLAG

accuracy WITH soil_moisture: 1.000


## 5. Prove it's leakage, don't just trust the number
A decision tree learns threshold rules. Print the rules it found —
you'll literally see it split on `soil_moisture <= ~54`. It re-discovered our label formula.

In [17]:
print(export_text(model, feature_names=feature_cols))

|--- soil_moisture <= 53.95
|   |--- class: 1
|--- soil_moisture >  53.95
|   |--- class: 0



## 6. Remove the leaked feature — watch it collapse
Now predict the SAME label using only humidity + temperature.
Those don't contain the answer, so the 'perfect' model has nothing to cheat with.

In [18]:
feature_cols_honest = ["air_humidity", "temperature"]   # soil removed
Xh = df[feature_cols_honest]
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    Xh, y, test_size=0.3, random_state=0)

model_h = DecisionTreeClassifier(max_depth=3, random_state=0)
model_h.fit(Xh_train, yh_train)
acc_h = model_h.score(Xh_test, yh_test)
print(f"accuracy WITHOUT soil_moisture: {acc_h:.3f}")   # drops hard

# baseline: what if we just always guess the most common class?
baseline = y.value_counts(normalize=True).max()
print(f"dumb baseline (guess majority): {baseline:.3f}")

accuracy WITHOUT soil_moisture: 0.451
dumb baseline (guess majority): 0.508


## Takeaway
- ~1.00 was **not skill — it was leakage.** The label lived inside a feature.
- The honest model is barely above the dumb baseline → humidity+temp alone can't predict this label.
- Rule for the whole roadmap: **a near-perfect score is a red flag, not a trophy. Ask what leaked first.**

## Your turn
1. Change `max_depth=3` to `max_depth=1`. Does the leaked model still hit ~1.00? Why does depth 1 already suffice? (hint: how many thresholds does it need?)
2. In one sentence at the bottom: explain to yourself why testing on `X_train` instead of `X_test` would have hidden the problem.